# 0 - Helpers

In [ ]:
from generate_time_comparison import *

# 1 - Retrieve networks

In [ ]:
scenarios_baseline = [
    "baseline",
    "baseline-rps",
    "baseline-co2-price25",
    "baseline-co2-price50",
    "baseline-co2-price100"
]

scenarios_25 = [
    "baseline",
    "energy-match-25",
    "hourly-match-25-90",
    "hourly-match-25-95",
    "hourly-match-25-98",
    "hourly-match-25-99",
]

scenarios_noadd = [
    "baseline",
    "hourly-match-noadd-10-99",
    "hourly-match-noadd-50-99",
    "hourly-match-noadd-90-99",
]

scenarios_sensitivity_25 = [
    "hourly-match-25-99",
    "hourly-match-EU-25-99",
    "hourly-match-no-LDES-25-99",
    "hourly-match-no-clean-firm-25-99"
]

scenarios_sensitivity_co2_25 = [
    "hourly-match-25-99",
    "hourly-match-co2-price25-25-99",
    "hourly-match-co2-price50-25-99",
    "hourly-match-co2-price100-25-99"
]

years = [2025, 2030]
# Main scenarios
scenarios_50 = [
    "baseline",
    "energy-match-50",
    "hourly-match-50-90",
    "hourly-match-50-95",
    "hourly-match-50-98",
    "hourly-match-50-99",
]

scenarios_sensitivity_50 = [
    "hourly-match-50-99",
    "hourly-match-EU-50-99",
    "hourly-match-no-LDES-50-99",
    "hourly-match-no-clean-firm-50-99",
]

scenarios_sensitivity_co2_50 = [
    "hourly-match-50-99",
    "hourly-match-co2-price25-50-99",
    "hourly-match-co2-price50-50-99",
]

# For all scenarios with 4 timesteps
years = [2025, 2030, 2035, 2040]
scenarios_all = {
    "baseline": scenarios_baseline,
    "main_ci_25": scenarios_25,
    "sensitivity_noadd": scenarios_noadd,
    "sensitivity_tech_ci_25": scenarios_sensitivity_25,
    "sensitivity_co2_ci_25": scenarios_sensitivity_co2_25,  
}

# For all scenarios with 2 timesteps
# years = [2025, 2030]
# scenarios_all = {
#     "main_ci_50": scenarios_50,
#     "sensitivity_tech_ci_50": scenarios_sensitivity_50,
#     "sensitivity_co2_ci_50": scenarios_sensitivity_co2_50,  
# }

df_networks_all = {}

for group, scenarios in scenarios_all.items():

    # Build MultiIndex
    index = pd.MultiIndex.from_product(
        [years, scenarios],
        names=["year", "scenario"]
    )

    # Create empty DataFrame
    df_networks = pd.DataFrame(index=index, columns=["network"])

    # Fill it
    for year, sc in index:
        try:
            n = pypsa.Network(f"../results/{sc}/networks/base_s_39___{year}.nc")
        except:
            print(f"{sc}-{year} not availabe")
            continue
        n = prepare_network(n)
        n.name = f"{sc}-{year}"
        df_networks.loc[(year, sc), "network"] = n

        m = strip_network_GoO(n)
        m.name = "GoO-" + m.name
        df_networks.loc[(year, sc), "GoO"] = m

    df_networks = df_networks.dropna()
    df_networks_all[group] = df_networks

# Calculate figsize automatically based on number of years
figsize_bar = get_figsize_from_years(years, FIGSIZE_BAR)
figsize_heatmap = get_figsize_from_years(years, FIGSIZE_HEATMAP_MAP)
figsize_arrow = FIGSIZE_ARROW  # Fixed size for arrow plot

print(f"Using figsize for bar plots: {figsize_bar}")
print(f"Using figsize for heatmap plots: {figsize_heatmap}")
print(f"Using figsize for arrow plot: {figsize_arrow}")


# 2 - GO market impacts

In [ ]:
figures_to_generate = [
        'a', # derive_energy_mix
        'b', # derive_energy_mix_go
        'c', # derive_capacity_mix
        'd', # derive_capacity_mix_new
        'e1', # derive_storage_energy_capacity
        'e2', # derive_storage_power_capacity
        'f', # derive_total_system_cost
        'g', # derive_total_system_cost_new
        'h', # derive_go_market_revenue
        'i', # derive_marginal_price
        'j', # derive_co2_emissions
        'k', # derive_cfe_curtailment
        'l', # derive_cfe_utilization
        #'m', # derive_co2_abatement_cost
]

# Figures for all groups of scenarios
fig_path_system = f"figures/time_comparison/system"
fig_path_countries = f"figures/time_comparison/countries"
countries = [
    'system', 'AL', 'AT', 'BA', 'BE', 'BG', 'CH', 'CZ', 
    'DE', 'DK', 'EE', 'ES', 'FI', 'FR', 'GB', 'GR', 'HR', 
    'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'ME', 'MK', 'NL', 
    'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK', 'XK'
]

for group, df_networks in df_networks_all.items():
    if group == "baseline":
        figures_to_generate_sys = list(set(figures_to_generate) - set(['b', 'h']))
    else:
        figures_to_generate_sys = figures_to_generate
    
    print(f"----------------------------------Processing {group} scenarios")
    
    for country in countries:
        
        if country == "system":
            scope = "system"
            fig_path = fig_path_system + f"/{group}"
        else:
            scope = country_converter.CountryConverter().convert(country, to="short_name")
            fig_path = fig_path_countries + f"/{country}/{group}"
            

        print(f"------------------> Analysing result at {scope} level")
        _, _ = derive_all_figures(
            df_networks, 
            country=country,
            plot_fig=True, 
            save_fig=True, 
            fig_path=fig_path, 
            save_csv=True, 
            figures=figures_to_generate_sys,
            figsize_bar=figsize_bar,
            figsize_heatmap=figsize_heatmap,
            figsize_arrow=figsize_arrow
        )
        
        # plt.show()